In [1]:
import sqlite3


connection = sqlite3.connect("student.db")

cursor = connection.cursor()


In [170]:
connection.close()

In [17]:
query = """
CREATE TABLE Habits(
ID INTEGER PRIMARY KEY AUTOINCREMENT,
Desc TEXT NOT NULL,
Priority INTEGER NOT NULL,
Prefernces INTEGER NOT NULL,
Type TEXT NOT NULL CHECK (Type IN ('Health', 'Learning', 'Creativity', 'Productivity')),
Time TIME NOT NULL,
Remarks TEXT
)
"""
cursor.execute(query)

In [167]:
connection.commit()  # Commit any uncommitted changes
connection.rollback()  # Rollback if something went wrong


In [16]:
cursor.execute("""DROP TABLE HabitDays;""")
cursor.execute("DROP TABLE HabitTimes;")
cursor.execute("DROP TABLE HabitS;")


In [159]:
connection.commit()

In [2]:
query_2 = """CREATE TABLE HabitDays (
    HabitID INTEGER,
    Day TEXT CHECK (Day IN ('Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday')),
    FOREIGN KEY (HabitID) REFERENCES Habits(ID)
);"""
cursor.execute(query_2)
connection.commit() 

OperationalError: table HabitDays already exists

In [19]:
query_3 = """CREATE TABLE HabitTimes (
    HabitID INTEGER,
    No_of_days_Completed INTEGER NOT NULL,
    Total_no_of_days INTEGER NOT NULL,
    
    FOREIGN KEY (HabitID) REFERENCES Habits(ID)
);"""
cursor.execute(query_3)
connection.commit()

In [22]:
# Insert sample habits
habits_data = [
    ('Morning Exercise', 1, 5, 'Health', '06:00'," Good progress in morning routine"),
    ('Read Books', 2, 4, 'Learning', '20:00', 'Reading speed improving'),
    ('Practice Piano', 3, 3, 'Creativity', '15:00', 'Getting better with scales')
]

cursor.execute("INSERT INTO Habits (Desc, Priority, Prefernces, Type, Time, Remarks) VALUES (?, ?, ?, ?, ?,?)", habits_data[0])
cursor.execute("INSERT INTO Habits (Desc, Priority, Prefernces, Type, Time, Remarks) VALUES (?, ?, ?, ?, ?,?)", habits_data[1])
cursor.execute("INSERT INTO Habits (Desc, Priority, Prefernces, Type, Time, Remarks) VALUES (?, ?, ?, ?, ?,?)", habits_data[2])

# Insert habit days
habit_days = [
    (1, 'Monday'),
    (1, 'Wednesday'),
    (1, 'Friday'),
    (2, 'Tuesday'),
    (2, 'Thursday'),
    (2, 'Saturday'),
    (3, 'Monday'),
    (3, 'Thursday'),
    (3, 'Sunday')
]

cursor.executemany("INSERT INTO HabitDays (HabitID, Day) VALUES (?, ?)", habit_days)

# Insert habit tracking data
habit_times = [
    (1, 10, 30),
    (2, 15, 45),
    (3, 8, 20)
]

cursor.executemany("INSERT INTO HabitTimes (HabitID, No_of_days_Completed, Total_no_of_days) VALUES (?, ?, ?)", habit_times)

connection.commit()

In [4]:
from langchain.prompts import ChatPromptTemplate

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="deepseek-r1-distill-llama-70b",temperature=0)


In [15]:
prompt_Remarker = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant that helps generate a single, polished remark for a habit by combining older feedback with new observations. Your goal is to merge both remarks into a clear, meaningful sentence or paragraph. Avoid repeating information and make the result sound natural and coherent.
     Just directly give the remark without any additional explanation or context. The remark should be consice and to the point. and in 20 words or less"""),
    ("user", "Previous remarks: {Remarks}"),
    ("user", "New Remark: {text}"),
])
chain_remarker = prompt_Remarker | llm
def Remarker(text, HabitID): # Updates remarks for a habit in HabitTimes table, Note, habit already has a remark
    connection = sqlite3.connect("student.db")
    cursor = connection.cursor()
    cursor.execute("SELECT Remarks FROM Habits WHERE ID = ?", (HabitID,))
    row = cursor.fetchone()
    if row:
        Remarks = row[0]
    else :
        Remarks = " "
    raw_remark = chain_remarker.invoke({"text": text, "Remarks": Remarks})
    remark_content = raw_remark.content
    print("Remark Content:", remark_content)
    # Slice the content after </think>
    if "</think>" in remark_content:
        sliced_remark = remark_content.split("</think>", 1)[1].strip()
    else:
        sliced_remark = remark_content.strip()  # Fallback if </think> not found

    cursor.execute("UPDATE Habits SET Remarks = ? WHERE ID = ?", (sliced_remark, HabitID))
    connection.commit()
    cursor.execute("SELECT * FROM Habits")
    rows = cursor.fetchall()
    for row in rows:
        print(row)
    connection.close()


In [16]:

Remarker("New Task Came up", 1)

Remark Content: <think>
Okay, so I need to help the user by generating a single, polished remark for a habit. The user has provided previous remarks and a new remark, and they want me to combine them into a concise sentence or paragraph without repeating information. The result should be clear, meaningful, and under 20 words.

Looking at the previous remarks: "A new urgent task has come up." And the new remark is: "New Task Came up." I notice that both are similar but phrased differently. The previous one is a full sentence, while the new one is more of a phrase.

I should find a way to merge these without repeating. Maybe I can take the essence of both. The previous remark mentions it's urgent, which is important. The new remark is shorter but still conveys the same message.

Perhaps I can combine the urgency with the new task. So, something like "An urgent new task has come up." That way, it's concise, includes the urgency, and is under 20 words.

Wait, let me check the word count. "

ProgrammingError: Cannot operate on a closed database.

In [99]:
results = cursor.fetchall()
for row in results:
    print(row)

(1, 'Morning Exercise', 1, 5, 'Health', '06:00', '<think>\nOkay, so the user has given me a task where I need to combine previous remarks with a new observation to create a concise remark. The')
(2, 'Read Books', 2, 4, 'Learning', '20:00', None)
(3, 'Practice Piano', 3, 3, 'Creativity', '15:00', None)
(4, 'Morning Exercise', 1, 5, 'Health', '06:00', ' Good progress in morning routine')
(5, 'Read Books', 2, 4, 'Learning', '20:00', 'Reading speed improving')
(6, 'Practice Piano', 3, 3, 'Creativity', '15:00', 'Getting better with scales')
(7, 'Morning Exercise', 1, 5, 'Health', '06:00', ' Good progress in morning routine')
(8, 'Read Books', 2, 4, 'Learning', '20:00', 'Reading speed improving')
(9, 'Practice Piano', 3, 3, 'Creativity', '15:00', 'Getting better with scales')
